[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-05-planning-decisions-and-bandits.ipynb)

# Advanced Discussion 5 — A* planning, act-ask-abort thresholds and bandit routing

Breadth-first search versus A* on a delivery domain, expected-value thresholds for acting, asking or aborting, and bandit strategies for routing.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure Python and NumPy; the bandit section takes about a minute.

In [ ]:
%pip install -q numpy

In [ ]:
import heapq, itertools, math, numpy as np
from collections import deque

## 1. classical planning: a delivery domain for Kestrel (STRIPS-style states and actions)

In [ ]:
GRAPH = {"depot": ["hub", "port"], "hub": ["depot", "city", "port"], "port": ["depot", "hub"], "city": ["hub", "airport"], "airport": ["city"]}
LOCS = list(GRAPH)
CAP = 2                                                        # the truck carries at most 2 parcels
def successors(state):
    truck, parcels = state                                     # parcels: tuple of locations, or 'T' when on the truck
    for nxt in GRAPH[truck]: yield ("drive->" + nxt, (nxt, parcels))
    onboard = sum(p == "T" for p in parcels)
    for i, p in enumerate(parcels):
        if p == truck and onboard < CAP: yield (f"load p{i}", (truck, parcels[:i] + ("T",) + parcels[i + 1:]))
        if p == "T": yield (f"unload p{i}", (truck, parcels[:i] + (truck,) + parcels[i + 1:]))
def solve(start, goal, method):
    def h(state):                                              # admissible heuristic: each parcel not at its goal needs at least 1 action (unload) - and 2 if it is still at its origin
        return sum(0 if p == g else (1 if p == "T" else 2) for p, g in zip(state[1], goal))
    frontier = [(h(start) if method == "astar" else 0, 0, start, [])] if method != "bfs" else deque([(start, [])])
    seen = {start: 0}; expanded = 0; tie = itertools.count()
    if method == "bfs":
        while frontier:
            state, plan = frontier.popleft(); expanded += 1
            if state[1] == goal: return plan, expanded
            for act, ns in successors(state):
                if ns not in seen: seen[ns] = 1; frontier.append((ns, plan + [act]))
        return None, expanded
    heap = [(h(start), next(tie), 0, start, [])]; best = {start: 0}
    while heap:
        f, _, g, state, plan = heapq.heappop(heap); expanded += 1
        if state[1] == goal: return plan, expanded
        for act, ns in successors(state):
            if ns not in best or g + 1 < best[ns]:
                best[ns] = g + 1; heapq.heappush(heap, ((g + 1 + h(ns)) if method == "astar" else g + 1, next(tie), g + 1, ns, plan + [act]))
    return None, expanded
problems = {
 "2 parcels": (("depot", ("depot", "hub")), ("airport", "airport")),
 "3 parcels": (("depot", ("depot", "hub", "port")), ("airport", "city", "airport")),
 "4 parcels": (("depot", ("depot", "hub", "port", "hub")), ("airport", "city", "hub", "airport")),
 "5 parcels": (("depot", ("depot", "hub", "port", "hub", "port")), ("airport", "city", "hub", "airport", "depot")),
}
print("problem     BFS (plan length, nodes)   A* with heuristic (plan length, nodes)")
for name, (start, goal) in problems.items():
    pb, nb = solve(start, goal, "bfs"); pa, na = solve(start, goal, "astar")
    print("%-10s  %2d, %6d                  %2d, %6d" % (name, len(pb), nb, len(pa), na))
print("plan for the 3-parcel problem:", solve(*problems["3 parcels"], "astar")[0])
# a naive 'always take the first applicable action toward a goal' agent
def greedy_agent(start, goal, max_steps=60):
    state = start
    for step in range(max_steps):
        if state[1] == goal: return step
        # picks the action whose successor has the smallest heuristic, ties broken by list order: can oscillate between equal states
        best = min(successors(state), key=lambda a: sum(0 if p == g else (1 if p == "T" else 2) for p, g in zip(a[1][1], goal)))
        state = best[1]
    return None
print("greedy one-step agent on the 3-parcel problem:", greedy_agent(*problems["3 parcels"]), "(None = never finished in 60 steps)")

## 2. deciding whether to act, ask a human, or give up: expected value under the agent's calibrated confidence p

In [ ]:
def action_values(p, c_error, c_ask=0.3, ask_accuracy=0.98, r_success=1.0):
    return {"act": p * r_success - (1 - p) * c_error,
            "ask": ask_accuracy * r_success - (1 - ask_accuracy) * c_error - c_ask,       # a human check is right 98% of the time but costs c_ask
            "abort": 0.0}
SYM = {"act": "A", "ask": "H", "abort": "x"}
print("expected value of each action by confidence (reward 1 for a correct action; c_ask = 0.3):")
grid = np.linspace(0.05, 0.95, 10)
print("cost of an error   " + " ".join("%4.2f" % p for p in grid) + "   act when confidence >=")
for ce in (2.0, 10.0, 50.0):
    row = []
    for p in grid:
        v = action_values(p, ce); row.append(SYM[max(v, key=v.get)])
    v0 = action_values(0.5, ce); alt = max(v0["ask"], 0.0)                         # best non-act option (constant in p)
    thr = (alt + ce) / (1 + ce)                                                    # act if p - (1-p) ce >= alt
    print("%15.1f    " % ce + "    ".join(row) + "     %.3f   (best fallback: %s)" % (thr, "ask a human" if v0["ask"] > 0 else "abort"))
print("A = act, H = ask a human, x = abort. Higher error costs raise the confidence you need before acting; if a human check costs more than it saves, abort is the fallback.")

## 3. bandits: learning which tool or model to route to when its success rate is unknown

In [ ]:
true_p = np.array([0.55, 0.65, 0.72])                          # three candidate routes; the best is the third
def run(strategy, T=2000, seed=0):
    r = np.random.RandomState(seed); n = np.zeros(3); s = np.zeros(3); regret = 0.0; a_ = np.ones(3); b_ = np.ones(3)
    for t in range(1, T + 1):
        if strategy == "greedy": arm = int(np.argmax(np.where(n == 0, 1e9, s / np.maximum(n, 1)))) if t > 3 else (t - 1) % 3
        elif strategy == "eps10": arm = r.randint(3) if r.rand() < 0.1 else int(np.argmax(s / np.maximum(n, 1)))
        elif strategy == "ucb": arm = int(np.argmax(np.where(n == 0, 1e9, s / np.maximum(n, 1) + np.sqrt(2 * math.log(t) / np.maximum(n, 1)))))
        elif strategy == "thompson": arm = int(np.argmax(r.beta(a_, b_)))
        x = float(r.rand() < true_p[arm]); n[arm] += 1; s[arm] += x; a_[arm] += x; b_[arm] += 1 - x; regret += true_p.max() - true_p[arm]
    return regret, n / T
print("\nrouting among 3 tools with success rates", true_p.tolist(), "- average over 300 runs of 2000 requests")
print("strategy            mean regret (lost successes)   share of traffic sent to the best tool")
for st in ("greedy", "eps10", "ucb", "thompson"):
    res = [run(st, seed=s) for s in range(300)]; print("%-18s  %8.1f                        %.2f" % (st, np.mean([r[0] for r in res]), np.mean([r[1][2] for r in res])))
print("always the best tool would lose 0; a fixed 33/33/33 split loses %.0f" % (2000 * (true_p.max() - true_p.mean())))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.